# Register Model

## Notebook Overview

- Install and Import Libraries
- Start Execution
- Configure Settings
- Log the Model to MLFlow
- Fetch the Latest Model Version from MLflow
- Load the Model and Run Inference

# Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()
logger.info("Notebook execution started.")

2025-09-08 19:29:03 - INFO - Notebook execution started.


# Install and Import Libraries

In [1]:
%%time

%pip install -r ../requirements.txt --quiet 

Note: you may need to restart the kernel to use updated packages.
CPU times: user 97 ms, sys: 41 ms, total: 138 ms
Wall time: 3.87 s


In [2]:
# -----------------------------
# Standard library imports
# -----------------------------
import json                 # JSON parsing and serialization
import os                   # Operating system utilities (paths, env vars, etc.)
import re                   # Regular expression utilities
import sys                  # Python runtime environment manipulation
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths
import time

# -----------------------------
# Data manipulation libraries
# -----------------------------
import numpy as np           # Numerical computations and arrays
import pandas as pd          # Data manipulation and analysis
from tabulate import tabulate  # Pretty-print tabular data

# -----------------------------
# Deep learning frameworks
# -----------------------------
import torch                 # PyTorch deep learning framework

# -----------------------------
# Experiment tracking (MLflow)
# -----------------------------
import mlflow                # ML lifecycle management and experiment tracking
import mlflow.pyfunc         # MLflow pyfunc interface for custom models
from mlflow import MlflowClient  # Client interface for MLflow operations
from mlflow.models.signature import ModelSignature  # Model signature definitions
from mlflow.tracking import MlflowClient           # Experiment tracking client
from mlflow.types.schema import (                  # Schema utilities for inputs/outputs
    ColSpec,
    ParamSchema,
    ParamSpec,
    Schema,
    TensorSpec,
)

# -----------------------------
# Local imports
# -----------------------------
# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import logger, download_from_s3_uri  # Project-specific utilities
from src.bert_recommendation_service import BERTTourismModel  # Custom BERT-based recommendation model

# Start Execution

In [3]:
import os
import json
import sys
from pathlib import Path
from datetime import datetime
import warnings
import re

import pandas as pd
import numpy as np
from tabulate import tabulate

# MLflow for Experiment Tracking and Model Management
import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec
from mlflow.tracking import MlflowClient
from nemo.collections.nlp.models.language_modeling import BERTLMModel



# # Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

# Import new MLflow models-from-code components
from src.mlflow import Logger
from src.mlflow.model import BERTModelWithHiddenStates
from src.utils import (
    load_config,
)

src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if src_path not in sys.path:
    sys.path.append(src_path)

# Correct import for onnx_utils
from src.onnx_utils import ModelExportConfig

import torch

[NeMo W 2025-09-08 19:29:14 nemo_logging:349] /opt/conda/envs/aistudio/lib/python3.10/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
      warnings.warn(
    


# Configure Settings

In [4]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [5]:
CORPUS_URI = f"s3://149536453923-hpaistudio-public-assets/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/corpus.csv"
EMBEDDINGS_URI = f"s3://149536453923-hpaistudio-public-assets/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/embeddings.csv"

CORPUS_PATH = "../data/raw/corpus.csv"
TOKENIZER_DIR = "../artifacts/tokenizer"
BERT_MODEL_NAME = "bert-large-uncased"
BERT_MODEL_DATAFABRIC_PATH = "/home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo"
EMBEDDINGS_OUTPUT_PATH = "../data/processed/"
BERT_MODEL_ONLINE_PATH = "/root/.cache/torch/NeMo/NeMo_1.22.0/bertlargeuncased/ca4ebba9f05a8ffb79845249ca046983/bertlargeuncased.nemo"
DEMO_PATH = "../demo"
EMBEDDINGS_PATH = "../data/processed/embeddings.csv"
CONFIG_PATH = "../configs/config.yaml"

# Define required constants for MLflow registration
EXPERIMENT_NAME = "BERT_Tourism_Experiment"
RUN_NAME = "BERT_Tourism_Run"
MODEL_NAME = "BERT_Tourism_Model"

In [6]:
%%time

saved_data = download_from_s3_uri(s3_uri=CORPUS_URI, local_path="../data/raw")
logger.info(f"Saved to: {saved_data}")

saved_data = download_from_s3_uri(s3_uri=EMBEDDINGS_URI, local_path="../data/processed")
logger.info(f"Saved to: {saved_data}")

CPU times: user 2.08 s, sys: 317 ms, total: 2.39 s
Wall time: 17 s


In [7]:
# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")

✅ Configuration loaded successfully


In [8]:
# Organize BERT artifacts into generic data structure for logging
def prepare_bert_data_for_logging(corpus_path, embeddings_path, tokenizer_dir):
    """
    Organize BERT-specific artifacts into the generic data/ directory structure
    expected by the universal loader.
    """
    import tempfile
    import shutil
    
    # Create temporary data directory
    temp_data_dir = os.path.join(tempfile.gettempdir(), "bert_data_artifacts")
    if os.path.exists(temp_data_dir):
        shutil.rmtree(temp_data_dir)
    os.makedirs(temp_data_dir)
    
    try:
        # Copy BERT artifacts to data directory
        if corpus_path and os.path.exists(corpus_path):
            shutil.copy2(corpus_path, os.path.join(temp_data_dir, "corpus.csv"))
            logger.info(f"Copied corpus to data structure: {corpus_path}")
        
        if embeddings_path and os.path.exists(embeddings_path):
            shutil.copy2(embeddings_path, os.path.join(temp_data_dir, "embeddings.csv"))
            logger.info(f"Copied embeddings to data structure: {embeddings_path}")
        
        if tokenizer_dir and os.path.exists(tokenizer_dir):
            shutil.copytree(tokenizer_dir, os.path.join(temp_data_dir, "tokenizer"))
            logger.info(f"Copied tokenizer to data structure: {tokenizer_dir}")
        
        return temp_data_dir
        
    except Exception as e:
        # Clean up on error
        if os.path.exists(temp_data_dir):
            shutil.rmtree(temp_data_dir)
        raise e

# Prepare BERT data structure
bert_data_path = prepare_bert_data_for_logging(CORPUS_PATH, EMBEDDINGS_PATH, TOKENIZER_DIR)

2025-09-08 19:29:24 - INFO - Copied corpus to data structure: ../data/raw/corpus.csv
2025-09-08 19:29:26 - INFO - Copied embeddings to data structure: ../data/processed/embeddings.csv
2025-09-08 19:29:26 - INFO - Copied tokenizer to data structure: ../artifacts/tokenizer


# Register and Log the Model to MLFlow

In [9]:
%%time

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))

# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

input_schema = Schema([ColSpec("string", "query")])
output_schema = Schema([
    TensorSpec(np.dtype("object"), (-1,), "List of Recommendations and Similarities")
])
params_schema = ParamSchema([ParamSpec("show_score", "boolean", False)])

# Define model signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema, params=params_schema)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Log the BERT Tourism model using generic Logger interface
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path="../configs/config.yaml",
        docs_path=bert_data_path,  # Use prepared BERT data structure
        model_path=BERT_MODEL_DATAFABRIC_PATH,
        demo_folder=DEMO_PATH
    )
    
    # Register the logged model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

2025-09-08 19:29:26 - INFO - Starting the experiment: BERT_Tourism_Experiment
2025-09-08 19:29:26 - INFO - Using MLflow tracking URI: /phoenix/mlflow
[NeMo W 2025-09-08 19:30:00 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    data_file: /home/yzhang/data/nlp/bert/47316/hdf5/lower_case_1_seq_len_512_max_pred_80_masked_lm_prob_0.15_random_seed_12345_dupe_factor_5_shard_1472_test_split_10/books_wiki_en_corpus/training/
    max_predictions_per_seq: 80
    batch_size: 16
    shuffle: true
    num_samples: -1
    num_workers: 2
    drop_last: false
    pin_memory: false
    
[NeMo W 2025-09-08 19:30:02 modelPT:617] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2025-09-08 19:30:02 modelPT:728] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 4.375e-05
        maximize: False
        weight_decay: 0.01
    )


[NeMo W 2025-09-08 19:30:02 lr_scheduler:890] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


[NeMo I 2025-09-08 19:30:03 save_restore_connector:249] Model BERTLMModel was successfully restored from /tmp/model_artifacts/models/bertlargeuncased.nemo.


2025-09-08 19:30:04 - INFO - 🔧 Generating ONNX model(s) for specified models...
2025-09-08 19:30:04 - INFO - 🔄 Converting pytorch model: bert_tourism_onnx
2025-09-08 19:30:04 - INFO - 📁 Model directory: bert_tourism_onnx
2025-09-08 19:30:04 - INFO - 🔍 Model identified as: pytorch
2025-09-08 19:30:04 - INFO - 🔄 Exporting loaded PyTorch model with opset 12...
2025-09-08 19:30:34 - INFO - ✅ PyTorch model exported to: bert_tourism_onnx/model.onnx
2025-09-08 19:30:34 - INFO - ✅ Converted bert_tourism_onnx to directory: bert_tourism_onnx
2025-09-08 19:30:34 - INFO - 📦 Added model directory artifact: model_directory -> bert_tourism_onnx
2025-09-08 19:30:34 - INFO -   No Triton structure requested, using model directories as-is
2025-09-08 19:31:05 - INFO - Model logged with artifacts: ['model_directory']
2025-09-08 19:31:05 - INFO - ✅ Model logged with model directory created!
Registered model 'BERT_Tourism_Model' already exists. Creating a new version of this model...
2025/09/08 19:31:10 WARN

CPU times: user 26.3 s, sys: 12.6 s, total: 38.9 s
Wall time: 1min 48s


# Fetch the Latest Model Version from MLflow

# Load the Model and Run Inference

In [10]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the "BERT_Tourism_Model" model (not yet in a specific stage)
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
latest_version = versions[0].version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

# Print the latest model version and its signature
print(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
print(f"Signature: {model_info.signature}")

Latest registered version of 'BERT_Tourism_Model': 12
Signature: inputs: 
  ['query': string (required)]
outputs: 
  ['List of Recommendations and Similarities': Tensor('object', (-1,))]
params: 
  ['show_score': boolean (default: False)]



In [11]:
%%time

# Load the trained BERT similarity model from MLflow
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
print(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

# Define a sample query for testing
query = "Give me a resort budget vacation suggestion"

# Use the model to predict similar results based on the query
result = model.predict({"query": [query]})

[NeMo W 2025-09-08 19:33:46 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    data_file: /home/yzhang/data/nlp/bert/47316/hdf5/lower_case_1_seq_len_512_max_pred_80_masked_lm_prob_0.15_random_seed_12345_dupe_factor_5_shard_1472_test_split_10/books_wiki_en_corpus/training/
    max_predictions_per_seq: 80
    batch_size: 16
    shuffle: true
    num_samples: -1
    num_workers: 2
    drop_last: false
    pin_memory: false
    
[NeMo W 2025-09-08 19:33:47 modelPT:617] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2025-09-08 19:33:47 modelPT:728] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 4.375e-05
        maximize: False
        weight_decay: 0.01
    )


[NeMo W 2025-09-08 19:33:47 lr_scheduler:890] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


[NeMo I 2025-09-08 19:33:48 save_restore_connector:249] Model BERTLMModel was successfully restored from /phoenix/mlflow/245450959274739937/models/m-756a73fd51eb411ea7878ac7bc41311e/artifacts/data/model_artifacts/models/bertlargeuncased.nemo.
Successfully loaded model 'BERT_Tourism_Model' version 12 for inference.
CPU times: user 31.6 s, sys: 10.5 s, total: 42.1 s
Wall time: 2min 28s


In [12]:
# Convert the result into a pandas DataFrame
df = pd.DataFrame(result)

# Drop unnecessary columns if needed
df = df.drop(columns=["Unnamed: 0", "Topic"], errors="ignore")

# Rename columns for better readability
df.rename(columns={"Pledge": "Recommended Option", "Similarity": "Relevance Score"}, inplace=True)

# Display the DataFrame in a tabular format
print(tabulate(df, headers="keys", tablefmt="fancy_grid"))

╒════╤═════════════════════════════════════════════════════════════════════════════════════════════════════╤═══════════════════╕
│    │ Recommended Option                                                                                  │   Relevance Score │
╞════╪═════════════════════════════════════════════════════════════════════════════════════════════════════╪═══════════════════╡
│  0 │ For a budget-friendly vacation, consider a resort with vacation options and cruise activities.      │          0.869201 │
├────┼─────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────┤
│  1 │ For a budget-friendly vacation, consider a getaway with beach options and vacation activities.      │          0.863931 │
├────┼─────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────┤
│  2 │ For a budget-friendly vacation, consider a getaway with hotel options and vacation activit

In [13]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-09-08 19:33:48 - INFO - ⏱️ Total execution time: 4m 45.34s
2025-09-08 19:33:48 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).